In [1]:
import re
from typing import Dict, List

# Departamentos válidos
DEPARTAMENTOS_VALIDOS = ['VEN', 'ADM', 'TEC', 'LOG', 'RRH']

# Series válidas
SERIES_VALIDAS = ['A', 'B', 'C', 'D', 'E']

In [2]:
def validar_producto(codigo: str) -> Dict:

    resultado = {
        "valido": False,
        "categoria": None,
        "numero": None,
        "pais": None
    }

    patron = r"^([A-Z]{3})-([0-9]{4})-([A-Z]{2})$"

    match = re.match(patron, codigo)

    if match:
        resultado["valido"] = True
        resultado["categoria"] = match.group(1)
        resultado["numero"] = match.group(2)
        resultado["pais"] = match.group(3)

    return resultado

In [3]:
print(validar_producto("TEC-0001-MX"))
print(validar_producto("tec-0001-MX"))

{'valido': True, 'categoria': 'TEC', 'numero': '0001', 'pais': 'MX'}
{'valido': False, 'categoria': None, 'numero': None, 'pais': None}


In [4]:
def validar_envio(codigo: str) -> Dict:

    resultado = {
        "valido": False,
        "fecha": None,
        "secuencial": None
    }

    patron = r"^ENV-(202[0-9]|2030)-(0[1-9]|1[0-2])-(0[1-9]|[12][0-9]|3[01])-([0-9]{6})$"

    match = re.match(patron, codigo)

    if match:

        anio = match.group(1)
        mes = match.group(2)
        dia = match.group(3)

        resultado["valido"] = True
        resultado["fecha"] = f"{anio}-{mes}-{dia}"
        resultado["secuencial"] = match.group(4)

    return resultado

In [5]:
print(validar_envio("ENV-2024-03-15-001234"))
print(validar_envio("ENV-2024-13-15-001234"))

{'valido': True, 'fecha': '2024-03-15', 'secuencial': '001234'}
{'valido': False, 'fecha': None, 'secuencial': None}


In [6]:
def validar_empleado(codigo: str) -> Dict:

    resultado = {
        "valido": False,
        "departamento": None,
        "numero": None
    }

    patron = r"^EMP-([A-Z]{3})-([1-9][0-9]{3})$"

    match = re.match(patron, codigo)

    if match:

        departamento = match.group(1)

        if departamento in DEPARTAMENTOS_VALIDOS:

            resultado["valido"] = True
            resultado["departamento"] = departamento
            resultado["numero"] = match.group(2)

    return resultado

In [7]:
print(validar_empleado("EMP-VEN-1234"))
print(validar_empleado("EMP-VEN-0123"))
print(validar_empleado("EMP-XXX-1234"))

{'valido': True, 'departamento': 'VEN', 'numero': '1234'}
{'valido': False, 'departamento': None, 'numero': None}
{'valido': False, 'departamento': None, 'numero': None}


In [8]:
def validar_factura(codigo: str) -> Dict:

    resultado = {
        "valido": False,
        "serie": None,
        "numero": None
    }

    patron = r"^FAC-([A-E])-([0-9]{6})$"

    match = re.match(patron, codigo)

    if match:

        resultado["valido"] = True
        resultado["serie"] = match.group(1)
        resultado["numero"] = match.group(2)

    return resultado

In [9]:
print(validar_factura("FAC-A-123456"))
print(validar_factura("FAC-F-123456"))

{'valido': True, 'serie': 'A', 'numero': '123456'}
{'valido': False, 'serie': None, 'numero': None}


In [10]:
def validar_codigo(codigo: str) -> Dict:

    resultado = {
        "codigo": codigo,
        "tipo": "desconocido",
        "valido": False,
        "detalles": {}
    }

    # ENVÍOS
    if codigo.startswith("ENV"):

        resultado["tipo"] = "envio"

        detalles = validar_envio(codigo)

    # EMPLEADOS
    elif codigo.startswith("EMP"):

        resultado["tipo"] = "empleado"

        detalles = validar_empleado(codigo)

    # FACTURAS
    elif codigo.startswith("FAC"):

        resultado["tipo"] = "factura"

        detalles = validar_factura(codigo)

    # PRODUCTOS
    else:

        detalles = validar_producto(codigo)

        if re.match(r"^[A-Za-z]{3}-", codigo):
            resultado["tipo"] = "producto"

    resultado["valido"] = detalles["valido"]
    resultado["detalles"] = detalles

    return resultado

In [11]:
print(validar_codigo("TEC-0001-MX"))
print(validar_codigo("ENV-2024-03-15-001234"))
print(validar_codigo("EMP-VEN-1234"))
print(validar_codigo("FAC-A-123456"))
print(validar_codigo("RANDOM-CODE"))

{'codigo': 'TEC-0001-MX', 'tipo': 'producto', 'valido': True, 'detalles': {'valido': True, 'categoria': 'TEC', 'numero': '0001', 'pais': 'MX'}}
{'codigo': 'ENV-2024-03-15-001234', 'tipo': 'envio', 'valido': True, 'detalles': {'valido': True, 'fecha': '2024-03-15', 'secuencial': '001234'}}
{'codigo': 'EMP-VEN-1234', 'tipo': 'empleado', 'valido': True, 'detalles': {'valido': True, 'departamento': 'VEN', 'numero': '1234'}}
{'codigo': 'FAC-A-123456', 'tipo': 'factura', 'valido': True, 'detalles': {'valido': True, 'serie': 'A', 'numero': '123456'}}
{'codigo': 'RANDOM-CODE', 'tipo': 'desconocido', 'valido': False, 'detalles': {'valido': False, 'categoria': None, 'numero': None, 'pais': None}}


In [12]:
def procesar_lote(codigos: List[str]) -> Dict:

    resultado = {
        "total": 0,
        "validos": 0,
        "invalidos": 0,
        "por_tipo": {
            "producto": {"total": 0, "validos": 0},
            "envio": {"total": 0, "validos": 0},
            "empleado": {"total": 0, "validos": 0},
            "factura": {"total": 0, "validos": 0},
            "desconocido": {"total": 0, "validos": 0}
        },
        "detalle": []
    }

    for codigo in codigos:

        r = validar_codigo(codigo)

        resultado["detalle"].append(r)

        resultado["total"] += 1

        tipo = r["tipo"]

        resultado["por_tipo"][tipo]["total"] += 1

        if r["valido"]:

            resultado["validos"] += 1
            resultado["por_tipo"][tipo]["validos"] += 1

        else:

            resultado["invalidos"] += 1

    return resultado

In [13]:
CODIGOS_PRUEBA = [

    # Productos
    "TEC-0001-MX",
    "ALI-9999-US",
    "tec-0001-MX",

    # Envíos
    "ENV-2024-03-15-001234",
    "ENV-2024-13-15-001234",

    # Empleados
    "EMP-VEN-1234",
    "EMP-VEN-0123",

    # Facturas
    "FAC-A-123456",
    "FAC-F-123456",

    # Desconocidos
    "RANDOM-CODE"
]

In [14]:
def mostrar_reporte(reporte: Dict):

    print("=" * 50)
    print("REPORTE DE VALIDACIÓN")
    print("=" * 50)

    print(f"\nTotal procesados: {reporte['total']}")
    print(f"Válidos: {reporte['validos']}")
    print(f"Inválidos: {reporte['invalidos']}")

    print("\nDesglose por tipo:")

    for tipo, datos in reporte["por_tipo"].items():

        print(f"{tipo}: {datos}")

In [15]:
reporte = procesar_lote(CODIGOS_PRUEBA)

mostrar_reporte(reporte)

REPORTE DE VALIDACIÓN

Total procesados: 10
Válidos: 5
Inválidos: 5

Desglose por tipo:
producto: {'total': 3, 'validos': 2}
envio: {'total': 2, 'validos': 1}
empleado: {'total': 2, 'validos': 1}
factura: {'total': 2, 'validos': 1}
desconocido: {'total': 1, 'validos': 0}
